# Predictors vs. measurement

The lab measured **kinetic** stability: residual activity over time at 50 °C.
The models in use predict **thermodynamic** stability (Tm, ΔΔG). These are not
the same quantity, and in the protein-engineering literature they do not
correlate: on an independent set of 51 mutants, Rosetta ΔΔG and FoldX showed no
significant correlation with the observed T50.

This notebook asks three questions of increasing interest.

**Q1 — Does the predictor rank the four variants the way A∞ does?**
With n = 4 there are 24 orderings, so a perfect match gives an exact
permutation p = 1/24 ≈ 0.042. That is the only outcome that reaches
significance; anything less is descriptive.

**Q2 — Does it put S126R last?**
This is the coarse signal: 22 % vs 60 %, with a clear structural explanation
(buried position, 12.9 Å from the iron, a charge forced into a packed pocket).
A predictor that misses this misses the easy case.

**Q3 — Does anything reproduce the non-additivity?**
The measurement says DOUBLE (48.4 %) behaves like D314A alone (47.0 %), not
like the sum of the two single effects. Most ΔΔG predictors for multiple
mutations are additive by construction. This is the question worth asking.

> Measurements come from digitized figure data (notebook 01), so every
> conclusion here is a hypothesis to confirm, not a result.

In [1]:
import sys, pathlib
from pathlib import Path

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import pandas as pd

from stability.sequences import SEQUENCES, verify_mutations
from stability.epistasis import epistasis, logit_energy, exact_rank_pvalue
from stability import plm

verify_mutations()

MUTATIONS = {
    "WT":     [],
    "D314A":  [(314, "D", "A")],
    "S126R":  [(126, "S", "R")],
    "DOUBLE": [(126, "S", "R"), (314, "D", "A")],
}
VARIANTS = list(MUTATIONS)

## 1. The measurement

`A_inf` is the plateau of residual activity. Notebook 01 established that it is
the robust observable here: `k` is identifiable for only two of the four
variants, and `t_half` does not exist for WT.

In [2]:
fits = pd.read_csv("../data/processed/inactivation_fits.csv")
measured = dict(zip(fits.variant, fits.a_inf))

print("A_inf (%) measured:")
for v in sorted(measured, key=lambda v: -measured[v]):
    print(f"   {v:7s} {measured[v]:5.1f}")

A_inf (%) measured:
   WT       60.0
   DOUBLE   48.4
   D314A    47.0
   S126R    22.1


### Putting the measurement on an additive scale

A∞ is a surviving fraction, and fractions do not add. Asking whether a double
mutant equals the sum of its singles only makes sense on a scale where effects
add. If the plateau reflects a partition between an active and an inactive
population, the natural scale is the logit, in units of RT:

$$G(A) = -\ln\frac{A}{100 - A}$$

**This is an assumption, not a measurement.** It treats the plateau as a
two-state equilibrium partition. That is consistent with the Sadana–Henley
picture and with the oligomer-dissociation hypothesis, but it is not
established. If the plateau is something else, the epistasis number below is
not meaningful either. Say so whenever it is quoted.

In [3]:
epi = epistasis(
    wt=measured["WT"],
    single_a=measured["D314A"],
    single_b=measured["S126R"],
    double=measured["DOUBLE"],
)

print(f"ddG D314A            = {epi['ddG_single_a']:+6.3f} RT")
print(f"ddG S126R            = {epi['ddG_single_b']:+6.3f} RT")
print(f"ddG DOUBLE, additive = {epi['ddG_double_additive']:+6.3f} RT"
      f"   ->  A_inf would be {epi['a_inf_double_if_additive']:5.1f} %")
print(f"ddG DOUBLE, observed = {epi['ddG_double_observed']:+6.3f} RT"
      f"   ->  A_inf observed {epi['a_inf_double_observed']:5.1f} %")
print()
print(f"MEASURED EPISTASIS   = {epi['epistasis']:+6.3f} RT")

ddG D314A            = +0.529 RT
ddG S126R            = +1.666 RT
ddG DOUBLE, additive = +2.195 RT   ->  A_inf would be  14.3 %
ddG DOUBLE, observed = +0.469 RT   ->  A_inf observed  48.4 %

MEASURED EPISTASIS   = -1.725 RT


## 2. ESM-2

Zero-shot scoring, two schemes. The difference between them is the point.

`wt_marginals` masks each mutated position in the wild-type sequence and sums
the single-mutant scores. It is **additive by construction** and cannot express
coupling, whatever the model knows.

`mutant_marginals` masks each position in a sequence that already carries the
other mutation, so the model sees the partner in context. This scheme **can**
express coupling. Whether it does is what we are testing.

In [4]:
MODEL = plm.DEFAULT_MODEL          # swap for plm.TOY_MODEL to smoke-test on CPU
DEVICE = "cuda"                    # "cpu" works too: only a handful of passes

tok, model, device = plm.load_model(MODEL, device=DEVICE)
print(f"loaded {MODEL} on {device}")

Loading weights: 100%|██████████| 539/539 [00:00<00:00, 1995.63it/s]


loaded facebook/esm2_t33_650M_UR50D on cuda


In [5]:
wt_seq = SEQUENCES["WT"]
scores = {}

for scheme in ("wt_marginals", "mutant_marginals"):
    scores[scheme] = {
        v: plm.score_variant(tok, model, device, wt_seq, MUTATIONS[v], scheme=scheme)
        for v in VARIANTS
    }

esm = pd.DataFrame(scores)
esm.index.name = "variant"
esm["a_inf_measured"] = [measured[v] for v in esm.index]
esm.round(3)

,wt_marginals,mutant_marginals,a_inf_measured
variant,,,
WT,0.000,0.000,60.042
D314A,-5.760,-5.760,46.964
S126R,-8.946,-8.946,22.123
DOUBLE,-14.706,-14.606,48.444


## Q1 — Ranking

Higher ESM-2 score means the model finds the variant more plausible; higher
A∞ means the lab found it more robust. If the model tracked the measurement,
the two orderings would agree.

In [6]:
rows = []
for scheme in ("wt_marginals", "mutant_marginals"):
    pred = [scores[scheme][v] for v in VARIANTS]
    meas = [measured[v] for v in VARIANTS]
    rho, p = exact_rank_pvalue(pred, meas, higher_is_better=True)
    rows.append({"scheme": scheme, "spearman_rho": round(rho, 3),
                 "exact_p": round(p, 4),
                 "significant_at_0.05": p < 0.05})

pd.DataFrame(rows)

,scheme,spearman_rho,exact_p,significant_at_0.05
0,wt_marginals,0.4,0.375,False
1,mutant_marginals,0.4,0.375,False


## Q2 — Is S126R ranked last?

The coarse check. S126R is far and away the worst variant in the lab
(22 % vs 47–60 %), and there is a structural reason for it.

In [7]:
for scheme in ("wt_marginals", "mutant_marginals"):
    order = sorted(VARIANTS, key=lambda v: -scores[scheme][v])
    worst = order[-1]
    print(f"{scheme:18s}  predicted order (best to worst): {' > '.join(order)}")
    print(f"{'':18s}  worst = {worst}   {'OK' if worst == 'S126R' else '<-- MISSES IT'}")
print()
print(f"{'measured':18s}  order: "
      f"{' > '.join(sorted(VARIANTS, key=lambda v: -measured[v]))}")

wt_marginals        predicted order (best to worst): WT > D314A > S126R > DOUBLE
                    worst = DOUBLE   <-- MISSES IT
mutant_marginals    predicted order (best to worst): WT > D314A > S126R > DOUBLE
                    worst = DOUBLE   <-- MISSES IT

measured            order: WT > DOUBLE > D314A > S126R


## Q3 — Non-additivity

The measured epistasis is large and positive: the double mutant is far better
than the sum of its parts.

`wt_marginals` scores the double as exactly the sum of its singles, so its
epistasis is 0 by construction — it cannot be anything else. It is included as
the null, not as a competitor.

`mutant_marginals` is free to deviate. The question is whether it does, and
whether the deviation has the right sign and anything like the right size.

In [8]:
rows = []
for scheme in ("wt_marginals", "mutant_marginals"):
    s = scores[scheme]
    additive = (s["D314A"] - s["WT"]) + (s["S126R"] - s["WT"])
    observed = s["DOUBLE"] - s["WT"]
    rows.append({
        "scheme": scheme,
        "pred_double_additive": round(additive, 4),
        "pred_double_actual": round(observed, 4),
        "pred_epistasis": round(observed - additive, 4),
    })

out = pd.DataFrame(rows)
out["measured_epistasis_RT"] = round(epi["epistasis"], 3)
out

,scheme,pred_double_additive,pred_double_actual,pred_epistasis,measured_epistasis_RT
0,wt_marginals,-14.7061,-14.7061,0.0000,-1.725
1,mutant_marginals,-14.7061,-14.6058,0.1003,-1.725


### Reading the result

Three outcomes are worth distinguishing, and they mean different things.

If `mutant_marginals` also gives ~0 epistasis, then ESM-2 treats the two
positions as independent. They sit 14.9 Å apart and 188 residues apart in
sequence, so this would be unsurprising — and it would make the point cleanly:
the model has no route to the effect the lab measured.

If it gives non-zero epistasis with the wrong sign, that is worse than nothing:
the model is confident and wrong.

If it gives the right sign, that is genuinely interesting and worth chasing
with the other predictors.

Whatever comes out, note the asymmetry: ESM-2 scores sequence plausibility, not
a free energy. Comparing its epistasis to a measured epistasis in RT units is a
comparison of rankings and signs, not of magnitudes. Do not report a ratio.

## 3. Optional: full-sequence pseudo-log-likelihood

A third scheme, fully non-additive: score every position of each variant with
that position masked, then take the difference against WT. One forward pass per
residue, so 423 per variant. Seconds on a GPU, minutes on CPU.

In [10]:
RUN_PLL = True   # set True when you want it

if RUN_PLL:
    pll = {v: plm.pseudo_log_likelihood(tok, model, device, 
                                        plm.apply_mutations(wt_seq, MUTATIONS[v]))
           for v in VARIANTS}
    dpll = {v: pll[v] - pll["WT"] for v in VARIANTS}

    additive = dpll["D314A"] + dpll["S126R"]
    print("delta-PLL vs WT:")
    for v in VARIANTS:
        print(f"   {v:7s} {dpll[v]:+9.3f}")
    print(f"\nadditive prediction for DOUBLE: {additive:+9.3f}")
    print(f"actual DOUBLE:                  {dpll['DOUBLE']:+9.3f}")
    print(f"epistasis:                      {dpll['DOUBLE'] - additive:+9.3f}")
else:
    print("skipped; set RUN_PLL = True to run")

delta-PLL vs WT:
   WT         +0.000
   D314A      -8.863
   S126R      -9.648
   DOUBLE    -18.501

additive prediction for DOUBLE:   -18.511
actual DOUBLE:                    -18.501
epistasis:                         +0.010


## 4. Predictors not run yet

Recorded so the gap is explicit rather than forgotten. ESM-2 is sequence-only;
every entry below brings information it does not have.

| Predictor | What it adds | Why not yet |
|---|---|---|
| SaProt | structure-aware (3Di tokens); used in the group's previous thesis | needs that code and a foldseek pass |
| TmProt | predicts Tm directly, the quantity at the centre of the mismatch | needs that code |
| ThermoMPNN | structure-based ΔΔG, strong baseline | install pending |
| ThermoMPNN-D | **has an explicit epistatic mode** — the direct test for Q3 | install pending; note its default coupling cutoff is 5 Å Cα–Cα and our positions are 14.9 Å apart, so it would fall back to additive by its own criterion |
| FoldX | classical physics-based baseline | academic licence and registration |

ThermoMPNN-D is the one that matters most for Q3: it is the only listed
predictor that even attempts coupling.

In [11]:
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

esm.to_csv(out_dir / "esm2_scores.csv")
out.to_csv(out_dir / "epistasis_comparison.csv", index=False)

print("saved esm2_scores.csv and epistasis_comparison.csv")
esm.round(3)

saved esm2_scores.csv and epistasis_comparison.csv


,wt_marginals,mutant_marginals,a_inf_measured
variant,,,
WT,0.000,0.000,60.042
D314A,-5.760,-5.760,46.964
S126R,-8.946,-8.946,22.123
DOUBLE,-14.706,-14.606,48.444
